# Concept Tutorial: Photon Streams

In [1]:
import fretbursts as smf

--------------------------------------------------------------
 You are running FRETBursts (version 1.0.1).

 If you use this software please cite the following paper:

   FRETBursts: An Open Source Toolkit for Analysis of Freely-Diffusing Single-Molecule FRET
   Ingargiola et al. (2016). http://dx.doi.org/10.1371/journal.pone.0160716 

--------------------------------------------------------------


In [2]:
raw = smf.photonHDF5.load("HP3_TE300_SPC630.hdf5")
data = smf.photonHDF5.regularize_dets(raw)

prd = smf.Param(smf.Periods, detdef=data.detdef, period=60.0)
bg = smf.Param(smf.BG, base=prd, func=smf.bg.exp_mlefit, tail_min=5e-4)
bursts = smf.Param(smf.Bursts, bg=bg, m=10, F=6.0, streams=smf.PhSel('0ex_1ex1em'))

## `frb.PhSel` objects

FRETBursts provides a number of tools for working with and specifying types of photons.

### Photon definitions

FRETBursts treats photons as belonging to different streams.

In ALEX/PIE measurments, we usually think of 3 "photon streams", 

1. Donor-excitation Donor-emission ($n_{Dex}^{Dem}$)
2. Donor-excitation Acceptor-emission ($n_{Dex}^{Aem}$)
3. Acceptor-excitation Acceptor-emission ($n_{Aex}^{Aem}$)

And we recognize that there is a 4th stream that should be blank, Acceptor-excitation Donor-emission ($n_{Aex}^{Dem}$)

If not using alternated excitation, only the 1 and 2 apply.

> **Note**
> 
> For clarity, when all characteristics of a photon are names, we will call this a stream (so in an ALEX setup, $n_{Dex}^{Dem}$ is a stream),
> while a given type, such as Donor-excitation is a channel.
> Excitation/emission/polarization/split are refered to as channel-types

If you have an MFD setup, each of the previously named streams get "split" into perpendicular and parrallel polarization channels.

Some setups also include beam splitters to stochastically split photons between 2 (or potentially even more) detectors, which could further add "split" channels.

Identification of particular streams, and groups of streams is achieved through the `frb.PhSel` objects.

### Deinfing a single stream

The `frb.PhSel` object is usually created from a string, where the desired channel selections are identified with indexes.

In keeping with Python, all indexing starts at 0.

For excitation and emission, the order of indexes is alway from shortest to longest wavelength.
For polarization, it is alway in order of increasing polarization angle, with parrallel being 0.
Split channels are arbitrary, and will depend on the setup.

The same `ex`, `em` `pol` and `split` abreviations are used.

So to define a donor-excitation-donor-emission stream we would specifiy:

In [3]:
sel_dd = smf.PhSel('0ex0em0split0pol')

So, with the above object, we can get the arrival times of all photons in the above selection in bursts:

In [4]:
data.get_column(smf.Column(bursts, 'ph_times', (sel_dd)))[:2]

array([array([1616772, 1619052, 1627704, 1627839, 1628419, 1628628, 1628732,
              1634173, 1634258, 1635461, 1641207, 1657978, 1662747, 1663029,
              1664333, 1664518, 1667449, 1667973, 1669621, 1670139, 1671041,
              1678301, 1678426, 1679243, 1680209, 1680375, 1680928, 1681697,
              1682985, 1684677])                                            ,
       array([2618305, 2622479, 2622610, 2623161, 2623177, 2623638, 2623745,
              2625232, 2638642, 2640588, 2640890, 2641469, 2641528, 2641780,
              2643532, 2644203, 2644821, 2644836, 2646642, 2662281, 2662650,
              2662673, 2664765, 2667626, 2679147, 2685593, 2685991, 2686425,
              2686675, 2688453, 2689930, 2690040, 2690854, 2693484, 2694494,
              2694601, 2695004, 2695200, 2724952, 2725450, 2728923])        ],
      dtype=object)

which is different from a donor-excitation-acceptor-emission

In [5]:
sel_da = smf.PhSel('0ex1em0split0pol')
data.get_column(smf.Column(bursts, 'ph_times', (sel_da)))[:2]

array([array([1612806, 1628104, 1662057]),
       array([2640790, 2642105, 2642309, 2686490])], dtype=object)

### `frb.PhSel` multiple streams

It is also possible to specify multiple streams that "span" several channels in a single `frb.PhSel` object.

This can be done by simply ommiting the channel-type from the definition, in which case all channels of that channel-type are included.
So for instance, the defintion below for an ALEX/PIE setup with no polarization and no split channels would cover both $n_{Dex}^{Dem}$ and $n_{Dex}^{Aem}$ streams (since there are only 1 pol and 1 split channel, omitting/all is the same as specifying 0\[channel type\]):

In [6]:
sel_d = smf.PhSel('0ex')
data.get_column(smf.Column(bursts, 'ph_times', (sel_dd)))[:2]

array([array([1616772, 1619052, 1627704, 1627839, 1628419, 1628628, 1628732,
              1634173, 1634258, 1635461, 1641207, 1657978, 1662747, 1663029,
              1664333, 1664518, 1667449, 1667973, 1669621, 1670139, 1671041,
              1678301, 1678426, 1679243, 1680209, 1680375, 1680928, 1681697,
              1682985, 1684677])                                            ,
       array([2618305, 2622479, 2622610, 2623161, 2623177, 2623638, 2623745,
              2625232, 2638642, 2640588, 2640890, 2641469, 2641528, 2641780,
              2643532, 2644203, 2644821, 2644836, 2646642, 2662281, 2662650,
              2662673, 2664765, 2667626, 2679147, 2685593, 2685991, 2686425,
              2686675, 2688453, 2689930, 2690040, 2690854, 2693484, 2694494,
              2694601, 2695004, 2695200, 2724952, 2725450, 2728923])        ],
      dtype=object)

Brackets can also be used to select multiple channels at once:

In [7]:
sel_daex = smf.PhSel('[0,1]ex')
data.get_column(smf.Column(bursts, 'ph_times', (sel_daex)))[:2]

array([array([1612806, 1616772, 1619052, 1627704, 1627839, 1628104, 1628419,
              1628628, 1628732, 1634173, 1634258, 1635461, 1641207, 1657978,
              1662057, 1662747, 1663029, 1664333, 1664518, 1667449, 1667973,
              1669621, 1670139, 1671041, 1675816, 1676141, 1678301, 1678426,
              1679243, 1680209, 1680375, 1680609, 1680699, 1680928, 1681697,
              1682985, 1684677, 1685840])                                   ,
       array([2614513, 2618305, 2622479, 2622610, 2623161, 2623177, 2623638,
              2623745, 2624644, 2625232, 2629890, 2638642, 2640588, 2640790,
              2640890, 2641469, 2641528, 2641780, 2642105, 2642309, 2643532,
              2644203, 2644821, 2644836, 2646642, 2662281, 2662650, 2662673,
              2664765, 2667626, 2679147, 2685593, 2685991, 2686425, 2686490,
              2686675, 2688453, 2689930, 2690040, 2690854, 2693484, 2694494,
              2694601, 2695004, 2695200, 2704384, 2707586, 2708523, 2724952

This is more useful for channel types with more than 2 channels:

In [8]:
sel_2ex = smf.PhSel('[1,2]ex') # cannot be used with current dataset, b/c too many channels

Note that for a setup without polarization or split channel-types, the two `PhSel` objects will behave the same 
(but not if there are polarization or split channel-types)

In [9]:
ph_sel_same = smf.PhSel('0ex0em'), smf.PhSel('0ex0em0pol0split')

Finally, it is possible to string several stream/stream combinations together, by separating them with `_` characters.

Below we have all "active" streams in a classic ALEX/PIE measurement:

In [10]:
sel_active = smf.PhSel('0ex_1ex1em')

## Set-like behavior of `frb.PhSel`

`frb.PhSel` objects behave like sets, with logical operations working as expected.
Note that since the `==` and `!=` are used to test if two `frb.PhSel` objects represent the *same* selection, 
the `@` and `^` operators are used in their respective places

### Logical operations

#### Inversion `frb.PhSel`

| `A = frb.PhSel('0em')` | 0ex | 1ex | 2ex | \| | `~A` | 0ex | 1ex | 2ex | \| | `B = frb.PhSel('0ex')` | 0ex | 1ex | 2ex | \| | `~B` | 0ex | 1ex | 2ex |
|------------------------|-----|-----|-----|----|------|-----|-----|-----|----|------------------------|-----|-----|-----|----|------|-----|-----|-----|
| 0em                    | -   | -   | -   | \| | 0em  |     |     |     | \| | 0em                    | -   |     |     | \| | 0em  |     | -   | -   |
| 1em                    |     |     |     | \| | 1em  | -   | -   | -   | \| | 1em                    | -   |     |     | \| | 1em  |     | -   | -   |
| 2em                    |     |     |     | \| | 2em  | -   | -   | -   | \| | 2em                    | -   |     |     | \| | 2em  |     | -   | -   |

#### `&` and `|` operations (aka `*` and `+`)

| `A = frb.PhSel('0em')` | 0ex | 1ex | 2ex | \| | `B = frb.PhSel('0ex')` | 0ex | 1ex | 2ex | \| | `A & B` | 0ex | 1ex | 2ex | \| | `A \| B` | 0ex | 1ex | 2ex |
|------------------------|-----|-----|-----|----|------------------------|-----|-----|-----|----|---------|-----|-----|-----|----|----------|-----|-----|-----|
| 0em                    | -   | -   | -   | \| | 0em                    | -   |     |     | \| | 0em     | -   |     |     | \| | 0em      | -   | -   | -   |
| 1em                    |     |     |     | \| | 1em                    | -   |     |     | \| | 1em     |     |     |     | \| | 1em      | -   |     |     |
| 2em                    |     |     |     | \| | 2em                    | -   |     |     | \| | 2em     |     |     |     | \| | 2em      | -   |     |     |


#### `@` and `^`

| `A = frb.PhSel('0em')` | 0ex | 1ex | 2ex | \| | `B = frb.PhSel('0ex')` | 0ex | 1ex | 2ex | \| | `A @ B` | 0ex | 1ex | 2ex | \| | `A ^ B` | 0ex | 1ex | 2ex |
|------------------------|-----|-----|-----|----|------------------------|-----|-----|-----|----|---------|-----|-----|-----|----|---------|-----|-----|-----|
| 0em                    | -   | -   | -   | \| | 0em                    | -   |     |     | \| | 0em     | -   |     |     | \| | 0em     |     | -   | -   |
| 1em                    |     |     |     | \| | 1em                    | -   |     |     | \| | 1em     |     | -   | -   | \| | 1em     | -   |     |     |
| 2em                    |     |     |     | \| | 2em                    | -   |     |     | \| | 2em     |     | -   | -   | \| | 2em     | -   |     |     |


### `frb.PhSel` are immutable, hashable, comparable

Here is a good oportunity to point out that much like `frb.GateGroup` objects,
`frb.PhSel` objects are immutable, hashable and comparable.

So you cannot change a `frb.PhSel` object after creation, all methods/operation that change a `frb.PhSel` object always are returning a copy.
Further the `==` and `!=` test if the two objects respresent the same set of streams.

In [11]:
selA = smf.PhSel('0ex') | smf.PhSel('1ex1em')
selB = smf.PhSel('0ex_1ex1em')
selA == selB

True

### all and none `PhSel`

You can also select all streams with `PhSel('all')` and no stream with `PhSel('none')`

In [12]:
smf.PhSel('all')

<class 'fretbursts.ph_sel.PhSel'>
<class 'fretbursts.ph_sel.PhStream'>
ex = all
em = all
pol = all
split = all

In [13]:
smf.PhSel('none')

<class 'fretbursts.ph_sel.PhSel'>

### Testing if one `smf.PhSel` is "in" another

Continuing the set-like behavor of `frb.PhSel` objects, it is possible to test wheter one `frb.PhSel` object is "in" another `frb.PhSel`.
Meaning that whatever streams the first `frb.PhSel` contains, the second.
Logically, we can thind of this as `A in B` is equivalent to `A & B == A`.

Below are some examples:

In [14]:
smf.PhSel('0ex1em_1ex1em') in smf.PhSel('0ex')

False

The above is `False`, since the first `smf.PhSel` contains streams from the `1ex` channel, which are not included in the second.

Reversing the order is also `False`, since there `smf.PhSel('0ex')` contains all streams with channel `0ex` while 
`smf.PhSel('0ex1em_1ex1em')`contains only `0ex1em`

In [15]:
smf.PhSel('0ex') in smf.PhSel('0ex1em_1ex1em')

False

However, if re remove the `1ex1em` stream from the definition, now we see that the contains operation returns `True`

In [16]:
smf.PhSel('0ex1em') in smf.PhSel('0ex')

True

This also works when we limit the second with some `em` channels, so long as they include `1em`

In [17]:
smf.PhSel('0ex1em') in smf.PhSel('0ex[0,1]ex')

True

If two `frb.PhSel` objects are the same, they will mutually return `True` with "in" operations:

In [18]:
smf.PhSel('1ex') in smf.PhSel('1ex')

True

## `smf.DetDef` objects

FRETBursts defines how a given setup sorts photons using the `frb.DetDef` class.
Here, the number of each channel-type is defined.
If a setup does not have a given channel-type, it has 1 channel for that channel-type.

> **Note**
>
> In most str and repr methods for objects defining streams, if a given channel-type is
> "not present" ie 1 for DetDef, or all in future cases, it is ommited.

All instances of `smf.photondata.PhotonData` objects have a `smf.photondata.PhotonData.detdef` attribute, which is a `smf.DetDef` object:

In [19]:
detdef = data.detdef
print(f'detdef string: "{detdef}", detdef.ex: {detdef.ex}, detdef.em: {detdef.em}, detdef.pol: {detdef.pol}, detdef.split: {detdef.split}')

detdef string: "DetDef2ex2em", detdef.ex: 2, detdef.em: 2, detdef.pol: 1, detdef.split: 1


`smf.DetDef` objects can be created directly simply by specify the number of a given stream as a keword argument.
All omitted channels are assumed to be 1:

In [20]:
detdef = smf.DetDef(ex=2, em=2)
detdef.ex, detdef.em, detdef.pol, detdef.split

(np.uint8(2), np.uint8(2), np.uint8(1), np.uint8(1))

`smf.DetDef` objects are immutable, hashable and comparable:

In [21]:
try:
    detdef.ex = 4
    print("detdef is mutable")
except AttributeError as e:
    print(f"detdef is immutable, setting raises {e}")

detdef is immutable, setting raises DetDef does not support assignment


In [22]:
detdef_dict = {detdef:"I'm here!"}
data.detdef == detdef, hash(detdef), detdef_dict[data.detdef]

(np.True_, -3449864020236994139, "I'm here!")

`DetDef` objects provide a way to map detectors indexes (those stored in the `PhotonData.dets` array) to photon streams.

This mapping goes both ways.
We can use the `DetDef.get_steram_ids()` method to convert a `frb.PhSel` object to an array of all detector ids that are "in" the `frb.PhSel`:

In [23]:
data.detdef.get_stream_ids(sel_dd)

array([0], dtype=uint8)

And from detector ids to `frb.PhSel` with the `frb.DetDef.stream_ids_to_PhSel()` method:

In [24]:
str(data.detdef.stream_ids_to_PhSel([2,3]))

'1ex'

## Negatively defined `frb.PhSel` channels

You may have wondered, it was shown that it is possible to "invert" a `frb.PhSel` object.
When the `frb.DetDef` is not know, this must be done using a negative definition.
That is that the `frb.PhSel` object defines not the channels it includes, but the channels that it excludes.

This can be done when specifying a `frb.PhSel` object using the `~` or `!` symbols in front of the index identifier:

In [25]:
sel_neg = smf.PhSel('~1ex')
sel_mneg = smf.PhSel('~[1,2]ex')

### `smf.PhSel.render_positive()` method

Negative definitions are useful for logical operations, 
but when the number of stream-types is known,
you often want to "look" at everything from the perspective of what streams are inclduded, and not based on both inclusion and exclusion.

Therefore the `smf.PhSel` object has a method: `smf.PhSel.render_positive()` which, has a single required argument: `detdef`
This must be a  `smf.DetDef`object, and will return a fully "postive" definition of the `smf.PhSel`.

In [26]:
str(sel_neg.render_positive(data.detdef))

'0ex[0,1]em0pol0split'

Note that this definition includes all channel-types, even though the pol and split are both unnecessary.

The `smf.PhSel.render_positive()` method has a single keyword-arguments: `covert_all=False`, when `True` it will
check if the definition includes any instances where the streams incldue all the channels in a given channel-type
and convert those to "all" definitions (which internally are actually negative definitions that don't include anything to exclude).

In [27]:
str(sel_neg.render_positive(data.detdef, convert_all=True))

'0ex'

## `smf.PhSel` internals

Internally, the `smf.PhSel` object is the end of a hierarchy.

This hierarchy starts with the `smf.ph_sel.ChannelSet` object.
This class allows positive and negative definition of streams, but does not define the channel-type (ie ex/em/pol/split).

The next level is the `smf.ph_sel.PhStream` which has 4 `smf.ph_sel.ChannelSet` objects as attributes, one for each channel-type.
This creates a "rectanglular" definition of a set of streams.
This can be represented by a frb.PhSel initializaton string that does not contains any underscores.
e.g. `"1ex0em"` and `"[0,2]ex~[1,2]em1pol~0split"` can be represented with a `smf.ph_sel.PhStream` object, 
but `"0ex_1ex1em"` would require 2 (`"0ex"` and `"1ex1em"` to be precise).

The `smf.PhSel` object combines all of these.
Internally, `smf.PhSel` has only a single attribute: `smf.PhSel.streams`, everything else is a property/method.
`smf.PhSel.streams` is a frozen set of `smf.ph_sel.PhStream` objects.

Functionally, any detector in one (or more) of the `smf.ph_sel.PhStream` objects in the `smf.PhSel.streams` attribute if a `smf.PhSel`, is in that `smf.PhSel`
(i.e. `smf.PhSel` is an `|` (or) gate of all the `smf.ph_sel.PhStream`s in its `smf.PhSel.streams` attribute).

In [28]:
streams = smf.PhSel('0ex_1ex1em').streams
for stream in streams:
    print(stream, type(stream))

0ex <class 'fretbursts.ph_sel.PhStream'>
[0,1]ex1em <class 'fretbursts.ph_sel.PhStream'>
